# Layer 1a — 系統異常偵測

目標：找出系統層面的異常訂單。同一 device 大部分訂單很快，但少數異常慢（排除 file_count 影響後）。
可能原因：SECS timeout、DB lock、queue stuck。

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv('../data/orders.csv')
df['order_created_at'] = pd.to_datetime(df['order_created_at'], format='%Y/%m/%d %I:%M:%S %p')
print(f"Total orders: {len(df)}")
print(f"Unique devices: {df['device_id'].nunique()}")


Total orders: 30000
Unique devices: 200


## 1. 計算 per-file duration 的 device-level 統計，標記 outliers

使用 `per_file_duration_avg_seconds` 做異常偵測（排除 file_count 影響）。
Outlier 定義：超過該 device 的 Q3 + 3×IQR。

In [2]:
# Compute per-device stats on per_file_duration_avg_seconds
device_stats = df.groupby('device_id')['per_file_duration_avg_seconds'].agg(
    median='median', q1=lambda x: x.quantile(0.25), q3=lambda x: x.quantile(0.75), count='count'
).reset_index()
device_stats['iqr'] = device_stats['q3'] - device_stats['q1']
device_stats['upper_fence'] = device_stats['q3'] + 3 * device_stats['iqr']

# Also check queue and device/db durations for outliers
for col, label in [('queue_duration_seconds', 'queue'),
                   ('device_duration_avg_seconds', 'device'),
                   ('db_duration_avg_seconds', 'db')]:
    stats = df.groupby('device_id')[col].agg(
        q1=lambda x: x.quantile(0.25), q3=lambda x: x.quantile(0.75)
    ).reset_index()
    stats[f'iqr_{label}'] = stats['q3'] - stats['q1']
    stats[f'upper_{label}'] = stats['q3'] + 3 * stats[f'iqr_{label}']
    device_stats = device_stats.merge(stats[['device_id', f'upper_{label}']], on='device_id')

# Merge thresholds back and flag outliers
df = df.merge(device_stats[['device_id', 'upper_fence', 'upper_queue', 'upper_device', 'upper_db']], on='device_id')
df['is_system_anomaly'] = (
    (df['per_file_duration_avg_seconds'] > df['upper_fence']) |
    (df['queue_duration_seconds'] > df['upper_queue']) |
    (df['device_duration_avg_seconds'] > df['upper_device']) |
    (df['db_duration_avg_seconds'] > df['upper_db'])
)

anomalies = df[df['is_system_anomaly']]
print(f"System anomalies: {len(anomalies)} / {len(df)} ({100*len(anomalies)/len(df):.1f}%)")
print(f"Devices with anomalies: {anomalies['device_id'].nunique()}")


System anomalies: 1576 / 30000 (5.3%)
Devices with anomalies: 200


## 2. 異常訂單的 Phase Breakdown 分析

判斷異常主要是哪個階段造成的。

In [3]:
# Classify anomaly type
PARALLELISM = 4

def classify_anomaly(row):
    reasons = []
    if row['queue_duration_seconds'] > row['upper_queue']:
        reasons.append('queue_stuck')
    if row['device_duration_avg_seconds'] > row['upper_device']:
        reasons.append('device_timeout')
    if row['db_duration_avg_seconds'] > row['upper_db']:
        reasons.append('db_lock')
    if row['per_file_duration_avg_seconds'] > row['upper_fence']:
        reasons.append('slow_processing')
    return ', '.join(reasons) if reasons else 'unknown'

anomalies = anomalies.copy()
anomalies['anomaly_type'] = anomalies.apply(classify_anomaly, axis=1)

# Phase breakdown for anomalies
anomalies['est_queue'] = anomalies['queue_duration_seconds']
anomalies['est_db'] = anomalies['db_duration_avg_seconds'] * anomalies['file_count'] / PARALLELISM
anomalies['est_device'] = anomalies['device_duration_avg_seconds'] * anomalies['file_count'] / PARALLELISM
anomalies['est_inner'] = anomalies['inner_processing_duration_avg_seconds'] * anomalies['file_count'] / PARALLELISM

# Which phase dominates?
phase_cols = ['est_queue', 'est_db', 'est_device', 'est_inner']
anomalies['dominant_phase'] = anomalies[phase_cols].idxmax(axis=1).str.replace('est_', '')

print("Anomaly type distribution:")
print(anomalies['anomaly_type'].value_counts().head(10))
print("\nDominant phase in anomalies:")
print(anomalies['dominant_phase'].value_counts())


Anomaly type distribution:
anomaly_type
queue_stuck                    1195
device_timeout                  193
db_lock                         178
queue_stuck, device_timeout       8
slow_processing                   2
Name: count, dtype: int64

Dominant phase in anomalies:
dominant_phase
queue     696
device    603
db        268
inner       9
Name: count, dtype: int64


## 3. 圖表

In [4]:
# Chart 1: Per-device per_file_duration distribution (top 20 devices by order count)
top_devices = df.groupby('device_id').size().nlargest(20).index
plot_df = df[df['device_id'].isin(top_devices)]

fig, ax = plt.subplots(figsize=(14, 6))
device_order = plot_df.groupby('device_id')['per_file_duration_avg_seconds'].median().sort_values(ascending=False).index
sns.boxplot(data=plot_df, x='device_id', y='per_file_duration_avg_seconds', order=device_order,
            flierprops={'marker': 'o', 'markersize': 3, 'alpha': 0.5}, ax=ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_title('Per-File Duration Distribution by Device (Top 20 by Order Count)')
ax.set_xlabel('Device ID')
ax.set_ylabel('Per-File Avg Duration (seconds)')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '1a_device_duration_boxplot.png', dpi=150)
plt.show()
print("Saved: 1a_device_duration_boxplot.png")


Saved: 1a_device_duration_boxplot.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65346/1005868800.py:9: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65346/1005868800.py:15: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [5]:
# Chart 2: Anomaly phase breakdown stacked bar
anomaly_type_counts = anomalies['anomaly_type'].str.split(', ').explode().value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: anomaly type counts
anomaly_type_counts.plot(kind='barh', ax=axes[0], color=sns.color_palette('Set2'))
axes[0].set_title('System Anomaly Type Distribution')
axes[0].set_xlabel('Count')

# Right: dominant phase
phase_counts = anomalies['dominant_phase'].value_counts()
colors = {'device': '#e74c3c', 'db': '#3498db', 'queue': '#f39c12', 'inner': '#2ecc71'}
phase_counts.plot(kind='bar', ax=axes[1], color=[colors.get(x, '#95a5a6') for x in phase_counts.index])
axes[1].set_title('Dominant Phase in Anomalous Orders')
axes[1].set_xlabel('Phase')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(REPORTS_DIR / '1a_anomaly_phase_breakdown.png', dpi=150)
plt.show()
print("Saved: 1a_anomaly_phase_breakdown.png")


Saved: 1a_anomaly_phase_breakdown.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65346/257159079.py:22: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [6]:
# Chart 3: Anomaly timeline scatter
fig, ax = plt.subplots(figsize=(14, 6))
normal = df[~df['is_system_anomaly']]
ax.scatter(normal['order_created_at'], normal['total_duration_seconds'],
           alpha=0.05, s=5, color='gray', label='Normal')
scatter = ax.scatter(anomalies['order_created_at'], anomalies['total_duration_seconds'],
                     alpha=0.6, s=15, c='red', label='System Anomaly')
ax.set_title('Order Duration Timeline (System Anomalies Highlighted)')
ax.set_xlabel('Order Created Time')
ax.set_ylabel('Total Duration (seconds)')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.savefig(REPORTS_DIR / '1a_anomaly_timeline.png', dpi=150)
plt.show()
print("Saved: 1a_anomaly_timeline.png")


Saved: 1a_anomaly_timeline.png


/var/folders/8w/j7kvkq4x17s8f3hvc8qz0gsw0000gq/T/ipykernel_65346/2483272442.py:15: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 4. 匯出異常標記供其他 notebook 使用

In [7]:
# Export anomaly flags
anomaly_flags = df[['order_id', 'is_system_anomaly']].copy()
anomaly_flags.to_csv('../data/system_anomaly_flags.csv', index=False)
print(f"Exported {anomaly_flags['is_system_anomaly'].sum()} system anomaly flags")

# Summary stats
print(f"\n=== Layer 1a Summary ===")
print(f"Total orders: {len(df)}")
print(f"System anomalies: {len(anomalies)} ({100*len(anomalies)/len(df):.1f}%)")
print(f"Top anomaly types:")
for t, c in anomalies['anomaly_type'].value_counts().head(5).items():
    print(f"  {t}: {c}")


Exported 1576 system anomaly flags

=== Layer 1a Summary ===
Total orders: 30000
System anomalies: 1576 (5.3%)
Top anomaly types:
  queue_stuck: 1195
  device_timeout: 193
  db_lock: 178
  queue_stuck, device_timeout: 8
  slow_processing: 2
